# Sobre os Datasets: Adult e Dry Bean

## Adult Dataset (Census Income)
O dataset **Adult** (também conhecido como Census Income) é um conjunto de dados clássico da UCI Machine Learning Repository. Ele contém informações demográficas e socioeconômicas de adultos dos EUA, extraídas do censo de 1994, e é amplamente utilizado para tarefas de classificação supervisionada.

- **Objetivo:** Prever se a renda anual de uma pessoa é superior a 50 mil dólares (`>50K`) ou não (`<=50K`).
- **Tamanho:** 48.842 instâncias (32.561 treino, 16.281 teste)
- **Variáveis:**
    - `age`: Idade
    - `workclass`: Tipo de emprego
    - `fnlwgt`: Peso final do censo
    - `education`: Nível de educação
    - `education-num`: Nível de educação (numérico)
    - `marital-status`: Estado civil
    - `occupation`: Ocupação
    - `relationship`: Relação familiar
    - `race`: Raça
    - `sex`: Sexo
    - `capital-gain`: Ganho de capital
    - `capital-loss`: Perda de capital
    - `hours-per-week`: Horas trabalhadas por semana
    - `native-country`: País de origem
    - `income`: **Rótulo alvo** (`<=50K` ou `>50K`)

## Dry Bean Dataset
O dataset **Dry Bean** contém características morfológicas extraídas de imagens de feijões secos de diferentes cultivares. É utilizado para classificação multiclasse.

- **Objetivo:** Classificar o tipo de feijão seco com base em atributos extraídos de imagens.
- **Tamanho:** 13.611 instâncias
- **Variáveis:**
    - 16 atributos numéricos extraídos de imagem, incluindo área, perímetro, comprimento, largura, coeficiente de excentricidade, etc.
    - `Class`: **Rótulo alvo** (tipo de feijão)
        - Possíveis valores:
            - `SEKER`
            - `BARBUNYA`
            - `BOMBAY`
            - `CALI`
            - `HOROZ`
            - `SIRA`
            - `DERMASON`

---
Estes datasets são baixados automaticamente e salvos na pasta `Trabalho03/data` para facilitar os experimentos de classificação.

In [2]:
import os
import urllib.request
import zipfile
import time
import socket
from urllib.error import URLError, HTTPError

data_dir = "data"
adult_dir = os.path.join(data_dir, "AdultDataset")
drybean_dir = os.path.join(data_dir, "DryBeanDataset")
os.makedirs(adult_dir, exist_ok=True)
os.makedirs(drybean_dir, exist_ok=True)

def baixar_arquivo_com_retry(url, destino, max_tentativas=3, delay=2):
    """
    Baixa um arquivo com tentativas de reconexão em caso de erro
    """
    if os.path.exists(destino):
        print(f"{os.path.basename(destino)} já existe.")
        return True
    
    for tentativa in range(max_tentativas):
        try:
            print(f"Baixando {os.path.basename(destino)}... (tentativa {tentativa + 1}/{max_tentativas})")
            req = urllib.request.Request(url)
            req.add_header('User-Agent', 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')
            with urllib.request.urlopen(req, timeout=30) as response:
                with open(destino, 'wb') as f:
                    chunk_size = 8192
                    while True:
                        chunk = response.read(chunk_size)
                        if not chunk:
                            break
                        f.write(chunk)
            print(f"✓ {os.path.basename(destino)} baixado com sucesso!")
            return True
        except (URLError, HTTPError, socket.error, ConnectionResetError) as e:
            print(f"✗ Erro na tentativa {tentativa + 1}: {str(e)}")
            if tentativa < max_tentativas - 1:
                print(f"Aguardando {delay} segundos antes da próxima tentativa...")
                time.sleep(delay)
                delay *= 2
            else:
                print(f"✗ Falha ao baixar {os.path.basename(destino)} após {max_tentativas} tentativas")
                return False
        except Exception as e:
            print(f"✗ Erro inesperado: {str(e)}")
            return False
    return False

# Baixar Adult dentro de data/AdultDataset
print("=== Baixando Dataset Adult ===")
adult_data_path = os.path.join(adult_dir, "adult.data")
adult_test_path = os.path.join(adult_dir, "adult.test")
adult_names_path = os.path.join(adult_dir, "adult.names")
adult_urls = [
    ("https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data", adult_data_path),
    ("https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test", adult_test_path),
    ("https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.names", adult_names_path)
]
adult_success = True
for url, path in adult_urls:
    if not baixar_arquivo_com_retry(url, path):
        adult_success = False
if adult_success:
    print("✓ Dataset Adult baixado completamente!")
else:
    print("✗ Alguns arquivos do dataset Adult falharam no download")

# Baixar DryBean considerando subpasta
print("\n=== Baixando Dataset DryBean ===")
drybean_xlsx_path = os.path.join(drybean_dir, "Dry_Bean_Dataset.xlsx")
drybean_arff_path = os.path.join(drybean_dir, "Dry_Bean_Dataset.arff")
drybean_txt_path = os.path.join(drybean_dir, "Dry_Bean_Dataset.txt")
drybean_zip_path = os.path.join(data_dir, "DryBeanDataset.zip")

if not (os.path.exists(drybean_xlsx_path) or os.path.exists(drybean_arff_path) or os.path.exists(drybean_txt_path)):
    if baixar_arquivo_com_retry("https://archive.ics.uci.edu/ml/machine-learning-databases/00602/DryBeanDataset.zip", drybean_zip_path):
        try:
            print("Extraindo arquivos do zip...")
            with zipfile.ZipFile(drybean_zip_path, 'r') as zip_ref:
                zip_ref.extractall(data_dir)
            print("Removendo arquivo zip...")
            os.remove(drybean_zip_path)
            print("✓ Dataset DryBean baixado e extraído com sucesso!")
        except Exception as e:
            print(f"✗ Erro ao extrair arquivo zip: {str(e)}")
    else:
        print("✗ Falha ao baixar dataset DryBean")
else:
    print("Arquivo do dataset DryBean já existe.")

print("\n=== Resumo ===")
print("Download dos datasets concluído.")

arquivos_verificar = [
    adult_data_path, adult_test_path, adult_names_path,
    drybean_xlsx_path, drybean_arff_path, drybean_txt_path
]
print("\nArquivos disponíveis:")
for arquivo in arquivos_verificar:
    if os.path.exists(arquivo):
        size = os.path.getsize(arquivo) / (1024 * 1024)
        print(f"✓ {os.path.basename(arquivo)} ({size:.1f} MB)")
    else:
        print(f"✗ {os.path.basename(arquivo)} (não encontrado)")

=== Baixando Dataset Adult ===
adult.data já existe.
adult.test já existe.
adult.names já existe.
✓ Dataset Adult baixado completamente!

=== Baixando Dataset DryBean ===
Arquivo do dataset DryBean já existe.

=== Resumo ===
Download dos datasets concluído.

Arquivos disponíveis:
✓ adult.data (3.8 MB)
✓ adult.test (1.9 MB)
✓ adult.names (0.0 MB)
✓ Dry_Bean_Dataset.xlsx (2.9 MB)
✓ Dry_Bean_Dataset.arff (3.6 MB)
✓ Dry_Bean_Dataset.txt (0.0 MB)
